# 04 Feature Engineering
Create text features using TF-IDF, word embeddings, and custom linguistic features.

In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from pathlib import Path

# Load cleaned data
DATA_PATH = '../data/cleaned_reviews.csv'
df = pd.read_csv(DATA_PATH, engine="python")
print(f'Loaded {len(df)} reviews')

X = df['reviewText_clean']
y = df['sentiment']

print(f'Features: {len(X)}, Labels: {len(y)}')

Loaded 20370 reviews
Features: 20370, Labels: 20370


In [7]:
# TF-IDF vectorization
print('Creating TF-IDF vectors...')
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(X)
print(f'TF-IDF shape: {X_tfidf.shape}')
print(f'Number of features: {X_tfidf.shape[1]}')

# Save vectorizer
VECTORIZER_PATH = '../models/tfidf_vectorizer.joblib'
Path(VECTORIZER_PATH).parent.mkdir(parents=True, exist_ok=True)
joblib.dump(tfidf, VECTORIZER_PATH)
print(f'Vectorizer saved to {VECTORIZER_PATH}')

Creating TF-IDF vectors...
TF-IDF shape: (20370, 5000)
Number of features: 5000
Vectorizer saved to ../models/tfidf_vectorizer.joblib


In [8]:
# Top TF-IDF features
feature_names = np.array(tfidf.get_feature_names_out())
top_indices = X_tfidf.mean(axis=0).A1.argsort()[-20:][::-1]
top_features = feature_names[top_indices]
top_scores = X_tfidf.mean(axis=0).A1[top_indices]

print('Top 20 TF-IDF Features:')
for i, (feature, score) in enumerate(zip(top_features, top_scores), 1):
    print(f'{i:2d}. {feature:20s} {score:.4f}')

Top 20 TF-IDF Features:
 1. amazon               0.0460
 2. service              0.0302
 3. customer             0.0273
 4. delivery             0.0242
 5. customer service     0.0220
 6. get                  0.0203
 7. time                 0.0198
 8. order                0.0195
 9. prime                0.0194
10. never                0.0174
11. company              0.0162
12. item                 0.0159
13. items                0.0157
14. good                 0.0157
15. one                  0.0153
16. refund               0.0153
17. always               0.0150
18. account              0.0149
19. day                  0.0148
20. great                0.0145


In [9]:
# Custom linguistic features
import re

def extract_linguistic_features(text):
    features = {}
    features['word_count'] = len(text.split())
    features['avg_word_length'] = np.mean([len(w) for w in text.split()]) if text else 0
    features['unique_word_ratio'] = len(set(text.split())) / max(len(text.split()), 1)
    features['punctuation_count'] = len([c for c in text if c in '!?.,;:'])
    features['uppercase_ratio'] = sum(c.isupper() for c in text) / max(len(text), 1)
    return features

# Extract features for all reviews
feature_list = []
for text in X:
    feature_list.append(extract_linguistic_features(text))

X_ling = pd.DataFrame(feature_list)
print('\nLinguistic Features:')
print(X_ling.describe())

# Save linguistic features
X_ling_path = '../data/linguistic_features.csv'
X_ling.to_csv(X_ling_path, index=False)
print(f'Linguistic features saved to {X_ling_path}')


Linguistic Features:
         word_count  avg_word_length  unique_word_ratio  punctuation_count  \
count  20370.000000     20370.000000       20370.000000            20370.0   
mean      41.793471         5.958441           0.887321                0.0   
std       44.178858         0.624076           0.099237                0.0   
min        1.000000         3.000000           0.116505                0.0   
25%       15.000000         5.600000           0.823529                0.0   
50%       29.000000         5.925926           0.900000                0.0   
75%       52.000000         6.272727           1.000000                0.0   
max      800.000000        35.000000           1.000000                0.0   

       uppercase_ratio  
count          20370.0  
mean               0.0  
std                0.0  
min                0.0  
25%                0.0  
50%                0.0  
75%                0.0  
max                0.0  
Linguistic features saved to ../data/linguistic_fe

In [10]:
# Combine features and save feature matrix
print('Combining feature matrices...')

# Convert sparse TF-IDF to dense for storage (selective rows)
X_tfidf_dense = X_tfidf.toarray()

# Combine: linguistic + TF-IDF
X_combined = np.hstack([X_ling.values, X_tfidf_dense])

print(f'Combined feature matrix shape: {X_combined.shape}')
print(f'Total features: {X_ling.shape[1]} linguistic + {X_tfidf_dense.shape[1]} TF-IDF')

# Save combined features
FEATURES_PATH = '../data/combined_features.npy'
np.save(FEATURES_PATH, X_combined)
LABELS_PATH = '../data/labels.npy'
np.save(LABELS_PATH, y.values)

print(f'Features and labels saved')
print(f'  - {FEATURES_PATH}')
print(f'  - {LABELS_PATH}')

Combining feature matrices...
Combined feature matrix shape: (20370, 5005)
Total features: 5 linguistic + 5000 TF-IDF
Features and labels saved
  - ../data/combined_features.npy
  - ../data/labels.npy
